# Ungraded Activity: Let's look at the Bag of Words model for SMS Spam/Ham classification
In this activity, we build upon our previous Bag of Words model for the SMS Spam/Ham classification problem. Let's load up the TF-IDF vectors we created in the previous demo, and then compute a kernel (similarity) matrix from the TF-IDF vectors. In this activity, we will:

* __Setup, Data, and Prerequisites:__ First, we set up the computational environment by including the `Include.jl` file and loading any needed resources, including loading the SMS spam/ham dataset and the pre-computed TF-IDF vectors from our previous work. 
* __Consensus Vectors for Spam and Ham Messages:__ Next, we'll create consensus vectors by averaging the TF-IDF vectors within each class (spam and ham).
* __Compute the similarity matrix between the consensus vectors:__ Finally, we'll compute a kernel similarity matrix to see how similar or different our spam and ham consensus vectors are to each other.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

In [13]:
include("Include.jl") # include the Include.jl file to set up the environment

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material. 

### Data
Let's load a public dataset of SMS messages that have been curated as either __spam__ or __ham__. The dataset we'll use is [publicly available on Kaggle](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset?resource=download) and is also discussed in the publications:
* Almeida, T. & Hidalgo, J. (2011). SMS Spam Collection [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CC84.
* Tiago A. Almeida, José María G. Hidalgo, and Akebo Yamakami. 2011. Contributions to the study of SMS spam filtering: new collection and results. In Proceedings of the 11th ACM symposium on Document engineering (DocEng '11). Association for Computing Machinery, New York, NY, USA, 259–262. https://doi.org/10.1145/2034691.2034742


We've packaged the spam/ham dataset in [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). We'll load the dataset using [the `MySMSSpamHamCorpus(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MySMSSpamHamCorpus) which returns [a `MySMSSpamHamRecordCorpusModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySMSSpamHamRecordCorpusModel) with the fields:
* The `records::Dict{Int, MySMSSpamHamRecordModel}` field holds the original records data as a dictionary, where the keys of the dictionary correspond to the headline index, and the values are [instances of the `MySMSSpamHamRecordModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySMSSpamHamRecordModel). Each record has the following fields:
    * `isspam`: has a value of `1` if the record is spam; otherwise, `0.`
    * `message`: the message of the SMS, unstructured text

* The `tokens::Dict{String, Int64}` field holds the vocabulary computed over the __entire dataset__ as a dictionary, where the dictionary's keys are the tokens (words) and the values of the index of the word. We assemble the `tokens` dictionary in alphabetical order. 
* The `inverse::Dict{Int64, String}` field is the inverse of the `tokens` dictionary, where the keys are the token indexes and the values are the tokens (words).

Let's call [the `MySMSSpamHamCorpus(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MySMSSpamHamCorpus) to load the spam/ham dataset and assign it to the `corpusmodel::MySMSSpamHamRecordCorpusModel` variable.

In [14]:
corpusmodel = MySMSSpamHamCorpus();

### TF-IDF Vectors
Next, we will load the TF-IDF vectors for the SMS messages in the corpus. We saved these in the `JLD2` file `my_sms_tfidf_vectors.jld2` in the previous demo. The TF-IDF vectors are stored in a dictionary, where the keys are the message indexes and the values are the TF-IDF vectors for each message. The TF-IDF vectors are stored as arrays of `Float64` values.

Let's load these vectors, and store them in the `tfidf_dictionary::Dict{Int64, Array{Float64,1}}` variable. The keys of the dictionary are the message indexes, and the values are the TF-IDF vectors for each message.


In [15]:
tfidf_dictionary = let

    # initialize -
    path_to_save_file = joinpath(_PATH_TO_DATA, "my_sms_tfidf_vectors.jld2"); # path to save the file
    tfidf_dictionary = load(path_to_save_file)["dataset"]; # load the tf-idf vectors from the file
end

Dict{Int64, Vector{Float64}} with 5573 entries:
  4986 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  4700 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  4576 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  2288 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  1703 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  1956 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  2350 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  3406 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  2841 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  2876 => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  687  => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, …
  185  => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  

### Balance
Before we proceed, let's check the balance of the dataset. The dataset is balanced if the number of spam messages is approximately equal to the number of ham messages. We can check this by counting the number of spam and ham messages in the `corpusmodel.records` dictionary.

> __TLDR:__ This data is not balanced, there are _way_ more ham messages than spam messages. While this is not a problem for the current analysis, it is something to keep in mind when interpreting the results.

Let's save the number of spam and ham messages in the `number_of_spam_messages::Int64` and `number_of_ham_messages::Int64` variables, respectively.

In [16]:
number_of_spam_messages = findall(r-> r.isspam == 1, corpusmodel.records) |> length
number_of_ham_messages = findall(r-> r.isspam == 0, corpusmodel.records) |> length

4826

We'll also compute the total number of messages in the dataset and store it in the `total_number_of_messages::Int64` variable. Finally, we'll print the fraction of spam messages in the dataset.

In [17]:
total_number_of_messages = length(corpusmodel.records);
println("Fraction of spam messages: ", number_of_spam_messages / total_number_of_messages);

Fraction of spam messages: 0.13403911717207967



## Task 1: Consensus Vectors for Spam and Ham Messages
In this task, we will compute the consensus vectors for the spam and ham messages in the corpus. The consensus vector is the average of the TF-IDF vectors for all messages in the corpus that are either spam or ham.

Let's start by computing the consensus vector for the spam messages. We'll do this by iterating records in the `corpusmodel.records` dictionary, and for each record, if the record is spam (i.e., `record.isspam == 1`), we'll add the TF-IDF vector for the record to a list of spam vectors. After iterating through all records, we'll compute the average of the spam vectors and store it in the `spam_consensus_vector::Array{Float64,1}` variable.

In [18]:
spam_consensus_vector = let

    # initialize -
    # Get the vector length from the first available vector
    first_key = first(keys(tfidf_dictionary))
    vector_length = length(tfidf_dictionary[first_key])
    
    spam_matrix = Array{Float64,2}(undef, number_of_spam_messages, vector_length); # create a matrix to store the spam vectors
    row_index = 1; # initialize the row index
    
    for (index, record) in corpusmodel.records # iterate through the records
        if record.isspam == 1 # if the record is spam
            spam_matrix[row_index, :] = tfidf_dictionary[index]; # add the TF-IDF vector to the spam matrix
            row_index += 1; # increment the row index
        end
    end

   # Compute mean and flatten to 1D vector
   spam_consensus_vector = vec(mean(spam_matrix, dims=1));
end

9998-element Vector{Float64}:
  0.0
  6.138265282331322e-5
  0.000168057070512098
  0.00011649026160753111
  0.00011649026160753111
  6.138265282331322e-5
  6.138265282331322e-5
  6.138265282331322e-5
  0.00011649026160753111
  0.0
  ⋮
  0.0
  0.0
  0.0
  0.0
 -1.0371125569776073e-6
 -1.0371125569776073e-6
  0.0
 -4.4771355982094425
  0.0

Now, let's do the same thing, but for ham messages. 

In [19]:
ham_consensus_vector = let

    # initialize -
    # Get the vector length from the first available vector
    first_key = first(keys(tfidf_dictionary))
    vector_length = length(tfidf_dictionary[first_key])
    
    ham_matrix = Array{Float64,2}(undef, number_of_ham_messages, vector_length); # create a matrix to store the ham vectors
    row_index = 1; # initialize the row index
    
    for (index, record) in corpusmodel.records # iterate through the records
        if record.isspam == 0 # if the record is ham
            ham_matrix[row_index, :] = tfidf_dictionary[index]; # add the TF-IDF vector to the ham matrix
            row_index += 1; # increment the row index
        end
    end

   # Compute mean and flatten to 1D vector
   ham_consensus_vector = vec(mean(ham_matrix, dims=1));
end

9998-element Vector{Float64}:
  3.361488888527909e-5
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  9.501210455659962e-6
  ⋮
  9.501210455659962e-6
  9.501210455659962e-6
  9.501210455659962e-6
  9.501210455659962e-6
 -1.0371125569776065e-6
 -1.0371125569776065e-6
  0.0
 -4.721541697113895
  5.488542399338351e-5

## Task 2: Compute the similarity matrix between the consensus vectors
In this task, we will compute the similarity between the spam and ham consensus vectors using a similarity metric, such as a kernel function. We'll pick a kernel function [exported by the `KernelFunctions.jl` package](https://github.com/JuliaGaussianProcesses/KernelFunctions.jl) to compute the similarity between the consensus vectors.

In [22]:
k = LinearKernel(); # use a linear kernel function from the KernelFunctions.jl package

Next, let's compute the kernel matrix $\mathbf{K}$, where the $(i,j)$-th entry of the matrix is the similarity between the $i$-th and $j$-th consensus vectors. We'll use the kernel function we selected in the previous step to compute the similarity.

> __Elements of the kernel matrix__: The kernel matrix is a square matrix where each element represents the similarity between two consensus vectors. The diagonal elements represent the self-similarity of each consensus vector, while the off-diagonal elements represent the similarity between different consensus vectors.

Let's compute the kernel matrix $\mathbf{K}$ using the kernel function we selected in the previous step. We'll save the resulting matrix as `K::Matrix{Float64}`.

In [21]:
K = let 
    
    # initialize -
    v₁ = spam_consensus_vector; # spam consensus vector
    v₂ = ham_consensus_vector; # ham consensus vector

    # compute the kernel matrix
    K = [
        k(v₁, v₁) k(v₁, v₂);
        k(v₂, v₁) k(v₂, v₂);
    ];

    K;
end

2×2 Matrix{Float64}:
 1.0       0.970414
 0.970414  1.0

### Analysis of the Kernel Matrix

Now let's take a look at what our kernel matrix $\mathbf{K}$ tells us. The matrix has four entries: $k_{11}$ shows how similar the spam consensus vector is to itself, $k_{22}$ does the same for the ham vector. The off-diagonal entries $k_{12}$ and $k_{21}$ tell us how similar the spam and ham vectors are to each other.

> __What to Look For__: If TF-IDF vectors were effectively capturing the differences between spam and ham messages, we'd expect the off-diagonal similarities to be noticeably different from the diagonal ones. The diagonal entries should be high (since each vector is most similar to itself), while the off-diagonal entries should be lower if spam and ham truly have distinct word usage patterns.

> __Interpreting the Results__: If all entries in the kernel matrix are surprisingly similar in magnitude, it suggests that our consensus vectors aren't capturing strong discriminative signals. This could happen because:
> 1. Simple averaging dilutes the discriminative features (which is what we did!)
> 2. Spam and ham messages share a lot of common vocabulary 
> 3. The imbalanced dataset (way more ham than spam) affects the consensus vectors

Think about it this way: if the kernel values show little contrast between spam and ham, it means our bag-of-words approach with simple averaging isn't finding clear linguistic differences. This motivates why we might need more sophisticated approaches like the weighted consensus or feature selection methods we explored later in the course.

The kernel matrix essentially gives us a quantitative measure of how well our current approach separates the two classes, and whether we need to try different strategies!

## Summary
In this activity, we explored how consensus vectors can reveal patterns in text classification by working with SMS spam and ham messages. We loaded our pre-computed TF-IDF vectors and discovered that our dataset is quite imbalanced, with way more ham messages than spam ones. 

By creating consensus vectors - essentially averaging all the TF-IDF vectors within each class - we distilled thousands of individual messages into just two representative vectors. The real magic happened when we computed the kernel similarity matrix between these consensus vectors, which gave us a quantitative way to see how similar or different spam and ham messages are in their word usage patterns.

This approach shows us that even with a simple bag-of-words model, we can start to understand the underlying structure in our text data through mathematical representations.